In [1]:
import numpy as np
import pandas as pd
import requests

from itertools import product

### Overall Code Logic

Data comes in as `bi_weekly_dates x hhs_region x variant`, with the corresponding `share`

End goal is to transform this into:
`daily_dates x fips` with `variant_1, variant_2, ..., variant_n` as features with their corresponding `shares` adding up to `1`

Where the `dates` will range from the beginning of the TLGRF project till now (when data gets updated)

### Import CDC's CSV File of COVID 19 Proportions 

In [2]:
url = "https://data.cdc.gov/api/views/jr58-6ysp/rows.csv?accessType=DOWNLOAD"
data_path = "../data/SARS-CoV-2_Variant_Proportions.csv"
UPDATE_DATA = True

if (UPDATE_DATA):
    response = requests.get(url)
    with open(data_path, "wb") as f:
        f.write(response.content)

df = pd.read_csv(data_path, low_memory=False)
df["week_ending"] = pd.to_datetime(df["week_ending"])
df["published_date"] = pd.to_datetime(df["published_date"])

df["variant"] = df["variant"].apply(str)
#df["share"] = df["share"].astype('int64')


df.sort_values(by=["week_ending","usa_or_hhsregion","variant"], inplace=True, ascending=[True,True,True])
df.head(30)

,usa_or_hhsregion,week_ending,variant,share,share_hi,share_lo,nchs_or_count_flag,modeltype,time_interval,published_date
21875,1,2021-01-02,B.1,0.215363,0.519812,0.065066,NaN,weighted,biweekly,2021-05-04
21876,1,2021-01-02,B.1.1.519,0.000000,NaN,NaN,NaN,weighted,biweekly,2021-05-04
21871,1,2021-01-02,B.1.1.7,0.000000,NaN,NaN,NaN,weighted,biweekly,2021-05-04
21870,1,2021-01-02,B.1.2,0.286312,0.422580,0.180268,NaN,weighted,biweekly,2021-05-04
21879,1,2021-01-02,B.1.234,0.031440,0.112799,0.008219,NaN,weighted,biweekly,2021-05-04
21878,1,2021-01-02,B.1.243,0.076162,0.154845,0.035769,NaN,weighted,biweekly,2021-05-04
21881,1,2021-01-02,B.1.351,0.000000,NaN,NaN,NaN,weighted,biweekly,2021-05-04
21874,1,2021-01-02,B.1.427,0.000000,NaN,NaN,NaN,weighted,biweekly,2021-05-04
21872,1,2021-01-02,B.1.429,0.001683,0.035162,0.000078,NaN,weighted,biweekly,2021-05-04
21882,1,2021-01-02,B.1.525,0.000000,NaN,NaN,NaN,weighted,biweekly,2021-05-04


In [3]:
unique_dates = pd.unique(df["week_ending"])
unique_dates

array(['2021-01-02T00:00:00.000000000', '2021-01-16T00:00:00.000000000',
       '2021-01-30T00:00:00.000000000', '2021-02-06T00:00:00.000000000',
       '2021-02-13T00:00:00.000000000', '2021-02-20T00:00:00.000000000',
       '2021-02-27T00:00:00.000000000', '2021-03-06T00:00:00.000000000',
       '2021-03-13T00:00:00.000000000', '2021-03-20T00:00:00.000000000',
       '2021-03-27T00:00:00.000000000', '2021-04-03T00:00:00.000000000',
       '2021-04-10T00:00:00.000000000', '2021-04-17T00:00:00.000000000',
       '2021-04-24T00:00:00.000000000', '2021-05-01T00:00:00.000000000',
       '2021-05-08T00:00:00.000000000', '2021-05-15T00:00:00.000000000',
       '2021-05-22T00:00:00.000000000', '2021-05-29T00:00:00.000000000',
       '2021-06-05T00:00:00.000000000', '2021-06-12T00:00:00.000000000',
       '2021-06-19T00:00:00.000000000', '2021-06-26T00:00:00.000000000',
       '2021-07-03T00:00:00.000000000', '2021-07-10T00:00:00.000000000',
       '2021-07-17T00:00:00.000000000', '2021-07-24

In [4]:
unique_regions = pd.unique(df["usa_or_hhsregion"])
unique_regions

array(['1', '10', '2', '3', '4', '5', '6', '7', '8', '9', 'USA'],
      dtype=object)

In [5]:
unique_variants = sorted(pd.unique(df["variant"]))
unique_variants

['A.2.5',
 'AY.1',
 'AY.2',
 'AY.3',
 'B.1',
 'B.1.1',
 'B.1.1.194',
 'B.1.1.519',
 'B.1.1.529',
 'B.1.1.7',
 'B.1.2',
 'B.1.234',
 'B.1.243',
 'B.1.351',
 'B.1.427',
 'B.1.429',
 'B.1.525',
 'B.1.526',
 'B.1.526.1',
 'B.1.526.2',
 'B.1.596',
 'B.1.617',
 'B.1.617.1',
 'B.1.617.2',
 'B.1.617.3',
 'B.1.621',
 'B.1.621.1',
 'B.1.626',
 'B.1.628',
 'B.1.637',
 'BA.1.1',
 'BA.2',
 'BA.2.12.1',
 'BA.2.75',
 'BA.2.75.2',
 'BA.4',
 'BA.4.6',
 'BA.5',
 'BA.5.2.6',
 'BF.11',
 'BF.7',
 'BN.1',
 'BQ.1',
 'BQ.1.1',
 'Other',
 'P.1',
 'P.2',
 'R.1',
 'XBB',
 'XBB.1.5']

### Multiple Published Reports of `share` per `date`, `hhs_region`, `variant`. Keep by latest `publication_date`

In [6]:
df_latest = df.sort_values('published_date', ascending=False).groupby(["week_ending","usa_or_hhsregion","variant"]).first().reset_index()
df_latest

,week_ending,usa_or_hhsregion,variant,share,share_hi,share_lo,nchs_or_count_flag,modeltype,time_interval,published_date
0,2021-01-02,1,B.1,0.215363,0.519812,0.065066,NaN,weighted,biweekly,2021-05-04
1,2021-01-02,1,B.1.1.519,0.000000,NaN,NaN,NaN,weighted,biweekly,2021-05-04
2,2021-01-02,1,B.1.1.7,0.000000,NaN,NaN,NaN,weighted,biweekly,2021-05-04
3,2021-01-02,1,B.1.2,0.286312,0.422580,0.180268,NaN,weighted,biweekly,2021-05-04
4,2021-01-02,1,B.1.234,0.031440,0.112799,0.008219,NaN,weighted,biweekly,2021-05-04
...,...,...,...,...,...,...,...,...,...,...
27110,2022-12-31,USA,BQ.1,0.183221,0.258504,0.125420,NaN,smoothed,weekly,2022-12-30
27111,2022-12-31,USA,BQ.1.1,0.268548,0.364878,0.189282,NaN,smoothed,weekly,2022-12-30
27112,2022-12-31,USA,Other,0.000245,0.000462,0.000127,NaN,smoothed,weekly,2022-12-30
27113,2022-12-31,USA,XBB,0.035711,0.049730,0.025421,NaN,smoothed,weekly,2022-12-30


### Impute missing variants for each date and region, setting their share to 0

In [7]:
#df_imputed = pd.DataFrame(columns=["date", "hhs_region", "variant"])

df_weekly_dates = pd.DataFrame({"date":unique_dates})
df_regions = pd.DataFrame({"hhs_region": unique_regions})
df_variants = pd.DataFrame({"variant": unique_variants})

df_imputed = pd.merge(left=df_weekly_dates, right=df_regions, how="cross")
df_imputed = pd.merge(left=df_imputed, right=df_variants, how="cross")

#for date, region, variant in product(unique_dates, unique_regions, unique_variants):
#    df_imputed = df_imputed.append({"date": date, "hhs_region": region, "variant": variant}, ignore_index=True)

df_imputed = df_imputed.merge(df_latest, how="left", left_on=["date", "hhs_region", "variant"], right_on=["week_ending","usa_or_hhsregion", "variant"])
df_imputed.fillna(0, inplace=True)
df_imputed

,date,hhs_region,variant,week_ending,usa_or_hhsregion,share,share_hi,share_lo,nchs_or_count_flag,modeltype,time_interval,published_date
0,2021-01-02,1,A.2.5,0,0,0.000000,0.000000,0.000000,0.0,0,0,0
1,2021-01-02,1,AY.1,0,0,0.000000,0.000000,0.000000,0.0,0,0,0
2,2021-01-02,1,AY.2,0,0,0.000000,0.000000,0.000000,0.0,0,0,0
3,2021-01-02,1,AY.3,0,0,0.000000,0.000000,0.000000,0.0,0,0,0
4,2021-01-02,1,B.1,2021-01-02 00:00:00,1,0.215363,0.519812,0.065066,0.0,weighted,biweekly,2021-05-04 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...
56645,2022-12-31,USA,P.1,0,0,0.000000,0.000000,0.000000,0.0,0,0,0
56646,2022-12-31,USA,P.2,0,0,0.000000,0.000000,0.000000,0.0,0,0,0
56647,2022-12-31,USA,R.1,0,0,0.000000,0.000000,0.000000,0.0,0,0,0
56648,2022-12-31,USA,XBB,2022-12-31 00:00:00,USA,0.035711,0.049730,0.025421,0.0,smoothed,weekly,2022-12-30 00:00:00


### Normalize the shares

In [8]:
df_imputed_select = df_imputed[["date", "hhs_region", "variant", "share"]]
def normalize(x):
    return (x) / (x.sum())
df_grouped = df_imputed_select.groupby(["date","hhs_region"])
df_normalized = df_grouped["share"].transform(normalize)
df_imputed_select["normalized_share"] = df_normalized
df_imputed_select = df_imputed_select[~(df_imputed_select["hhs_region"] == "USA")]
df_imputed_select["hhs_region"] = df_imputed_select["hhs_region"].astype(int)
df_imputed_select

/home/zwang937/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/home/zwang937/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


,date,hhs_region,variant,share,normalized_share
0,2021-01-02,1,A.2.5,0.000000,0.000000
1,2021-01-02,1,AY.1,0.000000,0.000000
2,2021-01-02,1,AY.2,0.000000,0.000000
3,2021-01-02,1,AY.3,0.000000,0.000000
4,2021-01-02,1,B.1,0.215363,0.215363
...,...,...,...,...,...
56595,2022-12-31,9,P.1,0.000000,0.000000
56596,2022-12-31,9,P.2,0.000000,0.000000
56597,2022-12-31,9,R.1,0.000000,0.000000
56598,2022-12-31,9,XBB,0.075548,0.075548


### Verify that for each date, all the present variants sum up to 100%

In [9]:
for date in unique_dates:
    df_segmented_by_date = df_imputed[df_imputed["date"] == date]
    for region in pd.unique(df_segmented_by_date["hhs_region"]):
        df_segmented_by_date_and_region = df_segmented_by_date[df_segmented_by_date["hhs_region"]==region]
        sum_of_variant_shares = df_segmented_by_date_and_region["share"].sum()
        print("Date: {}, Region: {}, Total Sum of Normalized Shares {}".format(date, region, sum_of_variant_shares))
    print("\n")

Date: 2021-01-02T00:00:00.000000000, Region: 1, Total Sum of Normalized Shares 1.000000013620594
Date: 2021-01-02T00:00:00.000000000, Region: 10, Total Sum of Normalized Shares 1.000000016763808
Date: 2021-01-02T00:00:00.000000000, Region: 2, Total Sum of Normalized Shares 0.999999996274711
Date: 2021-01-02T00:00:00.000000000, Region: 3, Total Sum of Normalized Shares 0.999999976134859
Date: 2021-01-02T00:00:00.000000000, Region: 4, Total Sum of Normalized Shares 1.0000000027939668
Date: 2021-01-02T00:00:00.000000000, Region: 5, Total Sum of Normalized Shares 1.0000000164145608
Date: 2021-01-02T00:00:00.000000000, Region: 6, Total Sum of Normalized Shares 0.999999988824127
Date: 2021-01-02T00:00:00.000000000, Region: 7, Total Sum of Normalized Shares 1.000000004074537
Date: 2021-01-02T00:00:00.000000000, Region: 8, Total Sum of Normalized Shares 1.000000009313226
Date: 2021-01-02T00:00:00.000000000, Region: 9, Total Sum of Normalized Shares 1.000000011175871
Date: 2021-01-02T00:00:00.0

Date: 2021-07-31T00:00:00.000000000, Region: 5, Total Sum of Normalized Shares 1.177163247097269
Date: 2021-07-31T00:00:00.000000000, Region: 6, Total Sum of Normalized Shares 1.1746681902267269
Date: 2021-07-31T00:00:00.000000000, Region: 7, Total Sum of Normalized Shares 1.33717579054172
Date: 2021-07-31T00:00:00.000000000, Region: 8, Total Sum of Normalized Shares 1.080765366050174
Date: 2021-07-31T00:00:00.000000000, Region: 9, Total Sum of Normalized Shares 1.094520850212573
Date: 2021-07-31T00:00:00.000000000, Region: USA, Total Sum of Normalized Shares 1.1567898894481912


Date: 2021-08-07T00:00:00.000000000, Region: 1, Total Sum of Normalized Shares 1.128993288161694
Date: 2021-08-07T00:00:00.000000000, Region: 10, Total Sum of Normalized Shares 1.07524733643774
Date: 2021-08-07T00:00:00.000000000, Region: 2, Total Sum of Normalized Shares 1.1762843919319141
Date: 2021-08-07T00:00:00.000000000, Region: 3, Total Sum of Normalized Shares 1.177325129477878
Date: 2021-08-07T00:00:0

Date: 2022-02-19T00:00:00.000000000, Region: 1, Total Sum of Normalized Shares 0.999999984633177
Date: 2022-02-19T00:00:00.000000000, Region: 10, Total Sum of Normalized Shares 0.999999997089617
Date: 2022-02-19T00:00:00.000000000, Region: 2, Total Sum of Normalized Shares 1.00000000061118
Date: 2022-02-19T00:00:00.000000000, Region: 3, Total Sum of Normalized Shares 1.000000010360964
Date: 2022-02-19T00:00:00.000000000, Region: 4, Total Sum of Normalized Shares 1.000000014348188
Date: 2022-02-19T00:00:00.000000000, Region: 5, Total Sum of Normalized Shares 1.000000003346941
Date: 2022-02-19T00:00:00.000000000, Region: 6, Total Sum of Normalized Shares 1.000000005908077
Date: 2022-02-19T00:00:00.000000000, Region: 7, Total Sum of Normalized Shares 0.9999999906867739
Date: 2022-02-19T00:00:00.000000000, Region: 8, Total Sum of Normalized Shares 0.999999976949766
Date: 2022-02-19T00:00:00.000000000, Region: 9, Total Sum of Normalized Shares 1.000000020285369
Date: 2022-02-19T00:00:00.000

Date: 2022-09-03T00:00:00.000000000, Region: 7, Total Sum of Normalized Shares 0.99999998608837
Date: 2022-09-03T00:00:00.000000000, Region: 8, Total Sum of Normalized Shares 0.999999972875232
Date: 2022-09-03T00:00:00.000000000, Region: 9, Total Sum of Normalized Shares 0.99999999461579
Date: 2022-09-03T00:00:00.000000000, Region: USA, Total Sum of Normalized Shares 0.9999999709252712


Date: 2022-09-10T00:00:00.000000000, Region: 1, Total Sum of Normalized Shares 1.00000001816079
Date: 2022-09-10T00:00:00.000000000, Region: 10, Total Sum of Normalized Shares 0.999999977182598
Date: 2022-09-10T00:00:00.000000000, Region: 2, Total Sum of Normalized Shares 0.9999999941210268
Date: 2022-09-10T00:00:00.000000000, Region: 3, Total Sum of Normalized Shares 0.9999999755527831
Date: 2022-09-10T00:00:00.000000000, Region: 4, Total Sum of Normalized Shares 1.000000008469216
Date: 2022-09-10T00:00:00.000000000, Region: 5, Total Sum of Normalized Shares 0.999999994528479
Date: 2022-09-10T00:00:00

### For each date, for each HHS region, take the latest `published_date` if there are duplicates of a variant

In [10]:
for date in unique_dates:
    df_segmented_by_date = df_imputed_select[df_imputed_select["date"] == date]
    for region in pd.unique(df_segmented_by_date["hhs_region"]):
        df_segmented_by_date_and_region = df_segmented_by_date[df_segmented_by_date["hhs_region"]==region]
        sum_of_variant_shares = df_segmented_by_date_and_region["normalized_share"].sum()
        print("Date: {}, Region: {}, Total Sum of Normalized Shares {}".format(date, region, sum_of_variant_shares))
    print("\n")

Date: 2021-01-02T00:00:00.000000000, Region: 1, Total Sum of Normalized Shares 1.0
Date: 2021-01-02T00:00:00.000000000, Region: 10, Total Sum of Normalized Shares 1.0
Date: 2021-01-02T00:00:00.000000000, Region: 2, Total Sum of Normalized Shares 1.0
Date: 2021-01-02T00:00:00.000000000, Region: 3, Total Sum of Normalized Shares 0.9999999999999999
Date: 2021-01-02T00:00:00.000000000, Region: 4, Total Sum of Normalized Shares 1.0000000000000002
Date: 2021-01-02T00:00:00.000000000, Region: 5, Total Sum of Normalized Shares 1.0
Date: 2021-01-02T00:00:00.000000000, Region: 6, Total Sum of Normalized Shares 1.0
Date: 2021-01-02T00:00:00.000000000, Region: 7, Total Sum of Normalized Shares 1.0
Date: 2021-01-02T00:00:00.000000000, Region: 8, Total Sum of Normalized Shares 1.0
Date: 2021-01-02T00:00:00.000000000, Region: 9, Total Sum of Normalized Shares 1.0


Date: 2021-01-16T00:00:00.000000000, Region: 1, Total Sum of Normalized Shares 1.0
Date: 2021-01-16T00:00:00.000000000, Region: 10, Total

Date: 2021-04-10T00:00:00.000000000, Region: 1, Total Sum of Normalized Shares 1.0000000000000002
Date: 2021-04-10T00:00:00.000000000, Region: 10, Total Sum of Normalized Shares 0.9999999999999998
Date: 2021-04-10T00:00:00.000000000, Region: 2, Total Sum of Normalized Shares 1.0
Date: 2021-04-10T00:00:00.000000000, Region: 3, Total Sum of Normalized Shares 1.0
Date: 2021-04-10T00:00:00.000000000, Region: 4, Total Sum of Normalized Shares 1.0
Date: 2021-04-10T00:00:00.000000000, Region: 5, Total Sum of Normalized Shares 0.9999999999999999
Date: 2021-04-10T00:00:00.000000000, Region: 6, Total Sum of Normalized Shares 0.9999999999999999
Date: 2021-04-10T00:00:00.000000000, Region: 7, Total Sum of Normalized Shares 1.0
Date: 2021-04-10T00:00:00.000000000, Region: 8, Total Sum of Normalized Shares 0.9999999999999999
Date: 2021-04-10T00:00:00.000000000, Region: 9, Total Sum of Normalized Shares 1.0


Date: 2021-04-17T00:00:00.000000000, Region: 1, Total Sum of Normalized Shares 1.0
Date: 202

Date: 2021-08-28T00:00:00.000000000, Region: 4, Total Sum of Normalized Shares 1.0
Date: 2021-08-28T00:00:00.000000000, Region: 5, Total Sum of Normalized Shares 0.9999999999999999
Date: 2021-08-28T00:00:00.000000000, Region: 6, Total Sum of Normalized Shares 0.9999999999999999
Date: 2021-08-28T00:00:00.000000000, Region: 7, Total Sum of Normalized Shares 1.0
Date: 2021-08-28T00:00:00.000000000, Region: 8, Total Sum of Normalized Shares 1.0
Date: 2021-08-28T00:00:00.000000000, Region: 9, Total Sum of Normalized Shares 1.0000000000000002


Date: 2021-09-04T00:00:00.000000000, Region: 1, Total Sum of Normalized Shares 1.0
Date: 2021-09-04T00:00:00.000000000, Region: 10, Total Sum of Normalized Shares 0.9999999999999999
Date: 2021-09-04T00:00:00.000000000, Region: 2, Total Sum of Normalized Shares 0.9999999999999998
Date: 2021-09-04T00:00:00.000000000, Region: 3, Total Sum of Normalized Shares 1.0
Date: 2021-09-04T00:00:00.000000000, Region: 4, Total Sum of Normalized Shares 1.0
Date: 202

Date: 2021-11-20T00:00:00.000000000, Region: 2, Total Sum of Normalized Shares 1.0000000000000002
Date: 2021-11-20T00:00:00.000000000, Region: 3, Total Sum of Normalized Shares 1.0
Date: 2021-11-20T00:00:00.000000000, Region: 4, Total Sum of Normalized Shares 1.0
Date: 2021-11-20T00:00:00.000000000, Region: 5, Total Sum of Normalized Shares 1.0
Date: 2021-11-20T00:00:00.000000000, Region: 6, Total Sum of Normalized Shares 1.0
Date: 2021-11-20T00:00:00.000000000, Region: 7, Total Sum of Normalized Shares 1.0
Date: 2021-11-20T00:00:00.000000000, Region: 8, Total Sum of Normalized Shares 1.0
Date: 2021-11-20T00:00:00.000000000, Region: 9, Total Sum of Normalized Shares 1.0


Date: 2021-11-27T00:00:00.000000000, Region: 1, Total Sum of Normalized Shares 1.0
Date: 2021-11-27T00:00:00.000000000, Region: 10, Total Sum of Normalized Shares 1.0
Date: 2021-11-27T00:00:00.000000000, Region: 2, Total Sum of Normalized Shares 1.0
Date: 2021-11-27T00:00:00.000000000, Region: 3, Total Sum of Normaliz

Date: 2022-04-09T00:00:00.000000000, Region: 9, Total Sum of Normalized Shares 0.9999999999999999


Date: 2022-04-16T00:00:00.000000000, Region: 1, Total Sum of Normalized Shares 1.0
Date: 2022-04-16T00:00:00.000000000, Region: 10, Total Sum of Normalized Shares 0.9999999999999999
Date: 2022-04-16T00:00:00.000000000, Region: 2, Total Sum of Normalized Shares 1.0
Date: 2022-04-16T00:00:00.000000000, Region: 3, Total Sum of Normalized Shares 1.0
Date: 2022-04-16T00:00:00.000000000, Region: 4, Total Sum of Normalized Shares 1.0
Date: 2022-04-16T00:00:00.000000000, Region: 5, Total Sum of Normalized Shares 1.0
Date: 2022-04-16T00:00:00.000000000, Region: 6, Total Sum of Normalized Shares 1.0
Date: 2022-04-16T00:00:00.000000000, Region: 7, Total Sum of Normalized Shares 1.0
Date: 2022-04-16T00:00:00.000000000, Region: 8, Total Sum of Normalized Shares 1.0
Date: 2022-04-16T00:00:00.000000000, Region: 9, Total Sum of Normalized Shares 1.0


Date: 2022-04-23T00:00:00.000000000, Region: 1, Tota

Date: 2022-07-02T00:00:00.000000000, Region: 6, Total Sum of Normalized Shares 1.0
Date: 2022-07-02T00:00:00.000000000, Region: 7, Total Sum of Normalized Shares 0.9999999999999999
Date: 2022-07-02T00:00:00.000000000, Region: 8, Total Sum of Normalized Shares 1.0
Date: 2022-07-02T00:00:00.000000000, Region: 9, Total Sum of Normalized Shares 1.0


Date: 2022-07-09T00:00:00.000000000, Region: 1, Total Sum of Normalized Shares 1.0
Date: 2022-07-09T00:00:00.000000000, Region: 10, Total Sum of Normalized Shares 1.0
Date: 2022-07-09T00:00:00.000000000, Region: 2, Total Sum of Normalized Shares 0.9999999999999999
Date: 2022-07-09T00:00:00.000000000, Region: 3, Total Sum of Normalized Shares 1.0
Date: 2022-07-09T00:00:00.000000000, Region: 4, Total Sum of Normalized Shares 1.0
Date: 2022-07-09T00:00:00.000000000, Region: 5, Total Sum of Normalized Shares 1.0
Date: 2022-07-09T00:00:00.000000000, Region: 6, Total Sum of Normalized Shares 1.0
Date: 2022-07-09T00:00:00.000000000, Region: 7, Total 

Date: 2022-11-26T00:00:00.000000000, Region: 4, Total Sum of Normalized Shares 0.9999999999999998
Date: 2022-11-26T00:00:00.000000000, Region: 5, Total Sum of Normalized Shares 0.9999999999999999
Date: 2022-11-26T00:00:00.000000000, Region: 6, Total Sum of Normalized Shares 1.0000000000000002
Date: 2022-11-26T00:00:00.000000000, Region: 7, Total Sum of Normalized Shares 0.9999999999999999
Date: 2022-11-26T00:00:00.000000000, Region: 8, Total Sum of Normalized Shares 0.9999999999999999
Date: 2022-11-26T00:00:00.000000000, Region: 9, Total Sum of Normalized Shares 1.0


Date: 2022-12-03T00:00:00.000000000, Region: 1, Total Sum of Normalized Shares 1.0
Date: 2022-12-03T00:00:00.000000000, Region: 10, Total Sum of Normalized Shares 1.0
Date: 2022-12-03T00:00:00.000000000, Region: 2, Total Sum of Normalized Shares 0.9999999999999999
Date: 2022-12-03T00:00:00.000000000, Region: 3, Total Sum of Normalized Shares 0.9999999999999999
Date: 2022-12-03T00:00:00.000000000, Region: 4, Total Sum of N

### HHS to State Mapping

In [11]:
df_imputed_select

,date,hhs_region,variant,share,normalized_share
0,2021-01-02,1,A.2.5,0.000000,0.000000
1,2021-01-02,1,AY.1,0.000000,0.000000
2,2021-01-02,1,AY.2,0.000000,0.000000
3,2021-01-02,1,AY.3,0.000000,0.000000
4,2021-01-02,1,B.1,0.215363,0.215363
...,...,...,...,...,...
56595,2022-12-31,9,P.1,0.000000,0.000000
56596,2022-12-31,9,P.2,0.000000,0.000000
56597,2022-12-31,9,R.1,0.000000,0.000000
56598,2022-12-31,9,XBB,0.075548,0.075548


In [12]:
hhs_data_path = "../data/hhs_regions.csv"
hhs_df = pd.read_csv(hhs_data_path, low_memory=False)
hhs_region_state_df = hhs_df[["region_number", "state_or_territory"]]

In [13]:
df_variant_share_by_state = pd.merge(left=df_imputed_select, right=hhs_region_state_df, how='left', left_on="hhs_region", right_on="region_number")
df_variant_share_by_state = df_variant_share_by_state[["date","hhs_region", "state_or_territory", "variant", "normalized_share"]]
df_variant_share_by_state = df_variant_share_by_state.rename(columns={"state_or_territory":"state", "normalized_share":"normalized_variant_share"})
df_variant_share_by_state["variant"] = "Variant % " + df_variant_share_by_state["variant"]
df_variant_share_by_state = df_variant_share_by_state.pivot_table(index=["date","hhs_region","state"], columns="variant", values="normalized_variant_share").reset_index()
df_variant_share_by_state.to_csv("../data/normalized_variant_share_by_date_and_state.csv", index=False)

In [14]:
df_variant_share_by_state

variant,date,hhs_region,state,Variant % A.2.5,Variant % AY.1,Variant % AY.2,Variant % AY.3,Variant % B.1,Variant % B.1.1,Variant % B.1.1.194,...,Variant % BF.7,Variant % BN.1,Variant % BQ.1,Variant % BQ.1.1,Variant % Other,Variant % P.1,Variant % P.2,Variant % R.1,Variant % XBB,Variant % XBB.1.5
0,2021-01-02,1,Connecticut,0.0,0.0,0.0,0.0,0.215363,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.337252,0.0,0.0,0.0,0.000000,0.000000
1,2021-01-02,1,Maine,0.0,0.0,0.0,0.0,0.215363,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.337252,0.0,0.0,0.0,0.000000,0.000000
2,2021-01-02,1,Massachusetts,0.0,0.0,0.0,0.0,0.215363,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.337252,0.0,0.0,0.0,0.000000,0.000000
3,2021-01-02,1,New Hampshire,0.0,0.0,0.0,0.0,0.215363,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.337252,0.0,0.0,0.0,0.000000,0.000000
4,2021-01-02,1,Rhode Island,0.0,0.0,0.0,0.0,0.215363,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.337252,0.0,0.0,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6072,2022-12-31,9,Republic of Palau,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.025809,0.041958,0.309209,0.370465,0.000470,0.0,0.0,0.0,0.075548,0.091676
6073,2022-12-31,10,Alaska,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.034255,0.039383,0.222777,0.386609,0.000484,0.0,0.0,0.0,0.035565,0.182380
6074,2022-12-31,10,Idaho,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.034255,0.039383,0.222777,0.386609,0.000484,0.0,0.0,0.0,0.035565,0.182380
6075,2022-12-31,10,Oregon,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.034255,0.039383,0.222777,0.386609,0.000484,0.0,0.0,0.0,0.035565,0.182380


### State to County Mapping

In [15]:
fips_list_path = "../data/fips-list.csv"
fips_list = pd.read_csv(fips_list_path, low_memory=False)
fips_list

,fips,county,state
0,1001,Autauga,Alabama
1,1003,Baldwin,Alabama
2,1005,Barbour,Alabama
3,1007,Bibb,Alabama
4,1009,Blount,Alabama
...,...,...,...
3231,72153,Yauco,Puerto Rico
3232,78010,St. Croix,Virgin Islands
3233,78020,St. John,Virgin Islands
3234,78030,St. Thomas,Virgin Islands


In [16]:
df_variant_share_by_fips = pd.merge(left=fips_list, right=df_variant_share_by_state, how='left', on="state")
df_variant_share_by_fips.to_csv("../data/df_variant_share_by_fips.csv", index=False)
df_variant_share_by_fips

,fips,county,state,date,hhs_region,Variant % A.2.5,Variant % AY.1,Variant % AY.2,Variant % AY.3,Variant % B.1,...,Variant % BF.7,Variant % BN.1,Variant % BQ.1,Variant % BQ.1.1,Variant % Other,Variant % P.1,Variant % P.2,Variant % R.1,Variant % XBB,Variant % XBB.1.5
0,1001,Autauga,Alabama,2021-01-02,4.0,0.000000,0.0,0.0,0.0,0.053232,...,0.000000,0.000000,0.000000,0.000000,0.240098,0.00000,0.010343,0.00000,0.000000,0.000000
1,1001,Autauga,Alabama,2021-01-16,4.0,0.000000,0.0,0.0,0.0,0.052789,...,0.000000,0.000000,0.000000,0.000000,0.235318,0.00000,0.007352,0.00000,0.000000,0.000000
2,1001,Autauga,Alabama,2021-01-30,4.0,0.003891,0.0,0.0,0.0,0.026768,...,0.000000,0.000000,0.000000,0.000000,0.479922,0.00000,0.002132,0.00080,0.000000,0.000000
3,1001,Autauga,Alabama,2021-02-06,4.0,0.001265,0.0,0.0,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.858039,0.00000,0.000000,0.00000,0.000000,0.000000
4,1001,Autauga,Alabama,2021-02-13,4.0,0.000843,0.0,0.0,0.0,0.024583,...,0.000000,0.000000,0.000000,0.000000,0.501443,0.00035,0.005559,0.00205,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
332895,99999,New York City,New York,2022-12-03,2.0,0.000000,0.0,0.0,0.0,0.000000,...,0.037494,0.022823,0.339862,0.306883,0.000642,0.00000,0.000000,0.00000,0.041575,0.067101
332896,99999,New York City,New York,2022-12-10,2.0,0.000000,0.0,0.0,0.0,0.000000,...,0.024744,0.024744,0.270167,0.349265,0.002942,0.00000,0.000000,0.00000,0.068348,0.133780
332897,99999,New York City,New York,2022-12-17,2.0,0.000000,0.0,0.0,0.0,0.000000,...,0.021161,0.016263,0.238384,0.278654,0.000370,0.00000,0.000000,0.00000,0.050601,0.314298
332898,99999,New York City,New York,2022-12-24,2.0,0.000000,0.0,0.0,0.0,0.000000,...,0.012052,0.011100,0.163023,0.205172,0.000180,0.00000,0.000000,0.00000,0.039109,0.523029


### Fill in the missing days in between every 2 weeks with the previous week's values

In [17]:
date_range = pd.date_range(unique_dates[0],unique_dates[-1])
unique_fips = pd.unique(df_variant_share_by_fips["fips"])

In [18]:
df_dates = pd.DataFrame({"date":date_range})
df_fips = pd.DataFrame({"fips":unique_fips})
df_all_dates_fips = pd.merge(left=df_dates, right=df_fips, how="cross")
df_all_dates_fips_imputted = pd.merge(left=df_all_dates_fips, right=df_variant_share_by_fips, how="left", on=["date","fips"])
df_all_dates_fips_imputted = df_all_dates_fips_imputted.sort_values(by=["date", "fips"], ascending=[True,True])

# Group by FIPS, while being sorted by dates, and then forward fill
COVID_Variants_Normalized_Share_All_Dates_FIPS = df_all_dates_fips_imputted.fillna(df_all_dates_fips_imputted.groupby(["fips"]).ffill())

COVID_Variants_Normalized_Share_All_Dates_FIPS.to_csv("../data/COVID_Variants_Normalized_Share_All_Dates_FIPS.csv", index=False)
COVID_Variants_Normalized_Share_All_Dates_FIPS

,date,fips,county,state,hhs_region,Variant % A.2.5,Variant % AY.1,Variant % AY.2,Variant % AY.3,Variant % B.1,...,Variant % BF.7,Variant % BN.1,Variant % BQ.1,Variant % BQ.1.1,Variant % Other,Variant % P.1,Variant % P.2,Variant % R.1,Variant % XBB,Variant % XBB.1.5
0,2021-01-02,1001,Autauga,Alabama,4.0,0.0,0.0,0.0,0.0,0.053232,...,0.000000,0.000000,0.00000,0.000000,0.240098,0.0,0.010343,0.0,0.000000,0.000000
1,2021-01-02,1003,Baldwin,Alabama,4.0,0.0,0.0,0.0,0.0,0.053232,...,0.000000,0.000000,0.00000,0.000000,0.240098,0.0,0.010343,0.0,0.000000,0.000000
2,2021-01-02,1005,Barbour,Alabama,4.0,0.0,0.0,0.0,0.0,0.053232,...,0.000000,0.000000,0.00000,0.000000,0.240098,0.0,0.010343,0.0,0.000000,0.000000
3,2021-01-02,1007,Bibb,Alabama,4.0,0.0,0.0,0.0,0.0,0.053232,...,0.000000,0.000000,0.00000,0.000000,0.240098,0.0,0.010343,0.0,0.000000,0.000000
4,2021-01-02,1009,Blount,Alabama,4.0,0.0,0.0,0.0,0.0,0.053232,...,0.000000,0.000000,0.00000,0.000000,0.240098,0.0,0.010343,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2359039,2022-12-31,72153,Yauco,Puerto Rico,2.0,0.0,0.0,0.0,0.0,0.000000,...,0.005696,0.006282,0.09247,0.125267,0.000072,0.0,0.000000,0.0,0.025083,0.721664
2359040,2022-12-31,78010,St. Croix,Virgin Islands,2.0,0.0,0.0,0.0,0.0,0.000000,...,0.005696,0.006282,0.09247,0.125267,0.000072,0.0,0.000000,0.0,0.025083,0.721664
2359041,2022-12-31,78020,St. John,Virgin Islands,2.0,0.0,0.0,0.0,0.0,0.000000,...,0.005696,0.006282,0.09247,0.125267,0.000072,0.0,0.000000,0.0,0.025083,0.721664
2359042,2022-12-31,78030,St. Thomas,Virgin Islands,2.0,0.0,0.0,0.0,0.0,0.000000,...,0.005696,0.006282,0.09247,0.125267,0.000072,0.0,0.000000,0.0,0.025083,0.721664


### Obtain Dates of Project

In [19]:
dataF_path = "../data/augmented_us-counties-states_latest.csv"
dataF = pd.read_csv(dataF_path, low_memory=False)
dataF["date"] = pd.to_datetime(dataF["date"])
dataF

,fips,date,county,state,cases,deaths,datetime,days_from_start,rolled_cases,LAT,...,totalTestsAntigen,totalTestsPeopleAntibody,totalTestsPeopleAntigen,totalTestsPeopleViral,totalTestsPeopleViralIncrease,totalTestsViral,totalTestsViralIncrease,metrics.testPositivityRatio,metrics.vaccinationsInitiatedRatio,metrics.vaccinationsCompletedRatio
0,1001,2022-11-25,Autauga,Alabama,101,230,2022-11-25,1039,89.571429,32.532237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1001,2020-12-20,Autauga,Alabama,1006,44,2020-12-20,334,881.142857,32.532237,...,NaN,82020.0,NaN,1783890.0,8462.0,NaN,0.0,0.256,0.000,0.000
2,1001,2021-02-05,Autauga,Alabama,749,76,2021-02-05,381,802.571429,32.532237,...,NaN,140376.0,NaN,2149684.0,0.0,NaN,0.0,0.156,0.043,0.010
3,1001,2021-10-03,Autauga,Alabama,534,142,2021-10-03,621,620.285714,32.532237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.118,0.431,0.340
4,1001,2022-01-08,Autauga,Alabama,1195,162,2022-01-08,718,880.428571,32.532237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.435,0.487,0.388
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3132585,99999,2021-04-28,New York City,New York,56198,32461,2021-04-28,463,62164.428571,40.658566,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3132586,99999,2022-04-27,New York City,New York,48728,40184,2022-04-27,827,46966.142857,40.658566,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3132587,99999,2022-08-14,New York City,New York,78078,41322,2022-08-14,936,84349.571429,40.658566,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3132588,99999,2022-09-03,New York City,New York,49884,41616,2022-09-03,956,56297.571429,40.658566,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Back Fill from Beginning of Variants Dataset to Start of Project Date


In [20]:
project_start_date = dataF["date"].min()
COVID_variants_end_date = date_range[-1]
print("project_start_date: {}, COVID_variants_end_date: {}".format(project_start_date, COVID_variants_end_date))

backfill_date_range = pd.date_range(project_start_date, COVID_variants_end_date)
backfill_date_range

project_start_date: 2020-01-27 00:00:00, COVID_variants_end_date: 2022-12-31 00:00:00


DatetimeIndex(['2020-01-27', '2020-01-28', '2020-01-29', '2020-01-30',
               '2020-01-31', '2020-02-01', '2020-02-02', '2020-02-03',
               '2020-02-04', '2020-02-05',
               ...
               '2022-12-22', '2022-12-23', '2022-12-24', '2022-12-25',
               '2022-12-26', '2022-12-27', '2022-12-28', '2022-12-29',
               '2022-12-30', '2022-12-31'],
              dtype='datetime64[ns]', length=1070, freq='D')

In [21]:
df_backfill_dates = pd.DataFrame({"date":backfill_date_range})
df_fips = pd.DataFrame({"fips":unique_fips})
df_backfill_dates_fips = pd.merge(left=df_backfill_dates, right=df_fips, how="cross")
df_backfilled_COVID_Variants_Normalized_Share_All_Dates_FIPS = pd.merge(left=df_backfill_dates_fips, right=COVID_Variants_Normalized_Share_All_Dates_FIPS, how="left", on=["date","fips"])
df_backfilled_COVID_Variants_Normalized_Share_All_Dates_FIPS = df_backfilled_COVID_Variants_Normalized_Share_All_Dates_FIPS.sort_values(by=["date", "fips"], ascending=[True,True])

df_backfilled_COVID_Variants_Normalized_Share_All_Dates_FIPS = df_backfilled_COVID_Variants_Normalized_Share_All_Dates_FIPS.fillna(df_backfilled_COVID_Variants_Normalized_Share_All_Dates_FIPS.groupby(["fips"]).bfill())
df_backfilled_COVID_Variants_Normalized_Share_All_Dates_FIPS

,date,fips,county,state,hhs_region,Variant % A.2.5,Variant % AY.1,Variant % AY.2,Variant % AY.3,Variant % B.1,...,Variant % BF.7,Variant % BN.1,Variant % BQ.1,Variant % BQ.1.1,Variant % Other,Variant % P.1,Variant % P.2,Variant % R.1,Variant % XBB,Variant % XBB.1.5
0,2020-01-27,1001,Autauga,Alabama,4.0,0.0,0.0,0.0,0.0,0.053232,...,0.000000,0.000000,0.00000,0.000000,0.240098,0.0,0.010343,0.0,0.000000,0.000000
1,2020-01-27,1003,Baldwin,Alabama,4.0,0.0,0.0,0.0,0.0,0.053232,...,0.000000,0.000000,0.00000,0.000000,0.240098,0.0,0.010343,0.0,0.000000,0.000000
2,2020-01-27,1005,Barbour,Alabama,4.0,0.0,0.0,0.0,0.0,0.053232,...,0.000000,0.000000,0.00000,0.000000,0.240098,0.0,0.010343,0.0,0.000000,0.000000
3,2020-01-27,1007,Bibb,Alabama,4.0,0.0,0.0,0.0,0.0,0.053232,...,0.000000,0.000000,0.00000,0.000000,0.240098,0.0,0.010343,0.0,0.000000,0.000000
4,2020-01-27,1009,Blount,Alabama,4.0,0.0,0.0,0.0,0.0,0.053232,...,0.000000,0.000000,0.00000,0.000000,0.240098,0.0,0.010343,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3462515,2022-12-31,72153,Yauco,Puerto Rico,2.0,0.0,0.0,0.0,0.0,0.000000,...,0.005696,0.006282,0.09247,0.125267,0.000072,0.0,0.000000,0.0,0.025083,0.721664
3462516,2022-12-31,78010,St. Croix,Virgin Islands,2.0,0.0,0.0,0.0,0.0,0.000000,...,0.005696,0.006282,0.09247,0.125267,0.000072,0.0,0.000000,0.0,0.025083,0.721664
3462517,2022-12-31,78020,St. John,Virgin Islands,2.0,0.0,0.0,0.0,0.0,0.000000,...,0.005696,0.006282,0.09247,0.125267,0.000072,0.0,0.000000,0.0,0.025083,0.721664
3462518,2022-12-31,78030,St. Thomas,Virgin Islands,2.0,0.0,0.0,0.0,0.0,0.000000,...,0.005696,0.006282,0.09247,0.125267,0.000072,0.0,0.000000,0.0,0.025083,0.721664


### Forward Fill from End of Variants Dataset to Current Date

In [22]:
current_date = dataF["date"].max()
print("Project Start Date: {}, Current Date for Update: {}".format(project_start_date, current_date))

forwardfill_date_range = pd.date_range(project_start_date, current_date)
forwardfill_date_range

Project Start Date: 2020-01-27 00:00:00, Current Date for Update: 2022-12-31 00:00:00


DatetimeIndex(['2020-01-27', '2020-01-28', '2020-01-29', '2020-01-30',
               '2020-01-31', '2020-02-01', '2020-02-02', '2020-02-03',
               '2020-02-04', '2020-02-05',
               ...
               '2022-12-22', '2022-12-23', '2022-12-24', '2022-12-25',
               '2022-12-26', '2022-12-27', '2022-12-28', '2022-12-29',
               '2022-12-30', '2022-12-31'],
              dtype='datetime64[ns]', length=1070, freq='D')

In [23]:
df_forwardfill_dates = pd.DataFrame({"date":forwardfill_date_range})
df_fips = pd.DataFrame({"fips":unique_fips})
df_forwardfill_dates_fips = pd.merge(left=df_forwardfill_dates, right=df_fips, how="cross")
df_ffill_bfill_COVID_Variants_Normalized_Share_All_Dates_FIPS = pd.merge(left=df_forwardfill_dates_fips, right=df_backfilled_COVID_Variants_Normalized_Share_All_Dates_FIPS, how="left", on=["date","fips"])
df_ffill_bfill_COVID_Variants_Normalized_Share_All_Dates_FIPS = df_ffill_bfill_COVID_Variants_Normalized_Share_All_Dates_FIPS.sort_values(by=["date", "fips"], ascending=[True,True])

df_ffill_bfill_COVID_Variants_Normalized_Share_All_Dates_FIPS = df_ffill_bfill_COVID_Variants_Normalized_Share_All_Dates_FIPS.fillna(df_ffill_bfill_COVID_Variants_Normalized_Share_All_Dates_FIPS.groupby(["fips"]).ffill())
df_ffill_bfill_COVID_Variants_Normalized_Share_All_Dates_FIPS.to_csv("../data/df_ffill_bfill_COVID_Variants_Normalized_Share_All_Dates_FIPS.csv", index=False)
df_ffill_bfill_COVID_Variants_Normalized_Share_All_Dates_FIPS

,date,fips,county,state,hhs_region,Variant % A.2.5,Variant % AY.1,Variant % AY.2,Variant % AY.3,Variant % B.1,...,Variant % BF.7,Variant % BN.1,Variant % BQ.1,Variant % BQ.1.1,Variant % Other,Variant % P.1,Variant % P.2,Variant % R.1,Variant % XBB,Variant % XBB.1.5
0,2020-01-27,1001,Autauga,Alabama,4.0,0.0,0.0,0.0,0.0,0.053232,...,0.000000,0.000000,0.00000,0.000000,0.240098,0.0,0.010343,0.0,0.000000,0.000000
1,2020-01-27,1003,Baldwin,Alabama,4.0,0.0,0.0,0.0,0.0,0.053232,...,0.000000,0.000000,0.00000,0.000000,0.240098,0.0,0.010343,0.0,0.000000,0.000000
2,2020-01-27,1005,Barbour,Alabama,4.0,0.0,0.0,0.0,0.0,0.053232,...,0.000000,0.000000,0.00000,0.000000,0.240098,0.0,0.010343,0.0,0.000000,0.000000
3,2020-01-27,1007,Bibb,Alabama,4.0,0.0,0.0,0.0,0.0,0.053232,...,0.000000,0.000000,0.00000,0.000000,0.240098,0.0,0.010343,0.0,0.000000,0.000000
4,2020-01-27,1009,Blount,Alabama,4.0,0.0,0.0,0.0,0.0,0.053232,...,0.000000,0.000000,0.00000,0.000000,0.240098,0.0,0.010343,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3462515,2022-12-31,72153,Yauco,Puerto Rico,2.0,0.0,0.0,0.0,0.0,0.000000,...,0.005696,0.006282,0.09247,0.125267,0.000072,0.0,0.000000,0.0,0.025083,0.721664
3462516,2022-12-31,78010,St. Croix,Virgin Islands,2.0,0.0,0.0,0.0,0.0,0.000000,...,0.005696,0.006282,0.09247,0.125267,0.000072,0.0,0.000000,0.0,0.025083,0.721664
3462517,2022-12-31,78020,St. John,Virgin Islands,2.0,0.0,0.0,0.0,0.0,0.000000,...,0.005696,0.006282,0.09247,0.125267,0.000072,0.0,0.000000,0.0,0.025083,0.721664
3462518,2022-12-31,78030,St. Thomas,Virgin Islands,2.0,0.0,0.0,0.0,0.0,0.000000,...,0.005696,0.006282,0.09247,0.125267,0.000072,0.0,0.000000,0.0,0.025083,0.721664


### Merge with `augmented_us-counties-states_latest.csv`

In [24]:
augmented_us = pd.merge(left=dataF, right=df_ffill_bfill_COVID_Variants_Normalized_Share_All_Dates_FIPS, on=["date","fips"], how="left")
augmented_us.to_csv(dataF_path, index=False)
augmented_us

,fips,date,county_x,state_x,cases,deaths,datetime,days_from_start,rolled_cases,LAT,...,Variant % BF.7,Variant % BN.1,Variant % BQ.1,Variant % BQ.1.1,Variant % Other,Variant % P.1,Variant % P.2,Variant % R.1,Variant % XBB,Variant % XBB.1.5
0,1001,2022-11-25,Autauga,Alabama,101,230,2022-11-25,1039,89.571429,32.532237,...,0.082753,0.030092,0.189461,0.246416,0.000000,0.000000,0.000000,0.000000,0.011752,0.000476
1,1001,2020-12-20,Autauga,Alabama,1006,44,2020-12-20,334,881.142857,32.532237,...,0.000000,0.000000,0.000000,0.000000,0.240098,0.000000,0.010343,0.000000,0.000000,0.000000
2,1001,2021-02-05,Autauga,Alabama,749,76,2021-02-05,381,802.571429,32.532237,...,0.000000,0.000000,0.000000,0.000000,0.479922,0.000000,0.002132,0.000800,0.000000,0.000000
3,1001,2021-10-03,Autauga,Alabama,534,142,2021-10-03,621,620.285714,32.532237,...,0.000000,0.000000,0.000000,0.000000,0.012255,0.000000,0.000000,0.000000,0.000000,0.000000
4,1001,2022-01-08,Autauga,Alabama,1195,162,2022-01-08,718,880.428571,32.532237,...,0.000000,0.000000,0.000000,0.000000,0.003592,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3132585,99999,2021-04-28,New York City,New York,56198,32461,2021-04-28,463,62164.428571,40.658566,...,0.000000,0.000000,0.000000,0.000000,0.116236,0.021284,0.000000,0.004324,0.000000,0.000000
3132586,99999,2022-04-27,New York City,New York,48728,40184,2022-04-27,827,46966.142857,40.658566,...,0.000000,0.000000,0.000000,0.000000,0.000934,0.000000,0.000000,0.000000,0.000000,0.000000
3132587,99999,2022-08-14,New York City,New York,78078,41322,2022-08-14,936,84349.571429,40.658566,...,0.003231,0.000000,0.000000,0.000000,0.001408,0.000000,0.000000,0.000000,0.000000,0.000000
3132588,99999,2022-09-03,New York City,New York,49884,41616,2022-09-03,956,56297.571429,40.658566,...,0.011722,0.000000,0.001689,0.000000,0.001727,0.000000,0.000000,0.000000,0.000000,0.000000
